# Week 5 - live coding (completed)

**Social and Cultural Analytics (7AAVBCS5)**

**Questions:** How are London venue records distributed, and how do recorded bike-hire durations vary by starting borough and weekday/weekend?

This notebook follows the lecture's code examples.

## Setup

Run this supplied cell. It provides pandas and the data loader; the bike file is loaded later, at its place in the lecture. You do not need to write or understand the loader function today.

In [1]:
import subprocess
from pathlib import Path

import pandas as pd

DATA_REPO = "https://github.com/markjhill/7AAVBCS5-data.git"


def data_path(name: str) -> Path:
    """Return a usable path to a module data file."""
    here = Path.cwd()
    for candidate in (here, *here.parents):
        for local in (candidate / "data" / name, candidate / name, candidate / Path(name).name):
            if local.exists():
                return local

    downloaded = Path("7AAVBCS5-data")
    if not downloaded.is_dir():
        print("No local data folder found -- downloading the module data (once)...")
        subprocess.run(
            ["git", "clone", "--depth", "1", DATA_REPO, str(downloaded)],
            check=True,
        )
    return downloaded / name

# Part 1: Categorical Data

## Categorical type

Use venues already loaded above. Converting to category records the possible labels; it does not change their definitions.

In [34]:
# Load venues 
venues = pd.read_csv(data_path("london_cultural_venues_2023.csv"))

In [35]:
# Convert the existing venue_type column to category and inspect its labels.
venues["venue_type"] = venues["venue_type"].astype("category")
print(venues["venue_type"].cat.categories)

Index(['Archives', 'Artists workspaces', 'Arts centres', 'Cinemas',
       'Commercial galleries', 'Creative coworking desk space',
       'Creative workspaces', 'Dance performance venues',
       'Dance rehearsal studios', 'Fashion and design', 'Jewellery design',
       'LGBTQ venues', 'Libraries', 'Makerspaces', 'Making and manufacturing',
       'Museums and public galleries', 'Music nightclubs',
       'Music recording studios', 'Music rehearsal studios',
       'Music venues all', 'Music venues grassroots',
       'Prop and costume making', 'Set and exhibition building',
       'Textile design', 'Theatre', 'Theatre rehearsal studios'],
      dtype='str')


## Notes 1 and 2: codes are not values and order is not assumed

Inspect the result before continuing - note the order in each case.

In [5]:
# Inspect internal codes and explicitly set an intended category order, following the slide.
likert = pd.Series(["Disagree", "Neutral", "Agree"]).astype("category")
print("internal codes:", list(likert.cat.codes))
print("string values:", list(likert.astype("string")))

numbers = pd.Series(["10", "20", "30"]).astype("category")
print("internal codes:", list(numbers.cat.codes))
print("numeric values:", list(numbers.astype("string").astype(int)))

internal codes: [1, 2, 0]
string values: ['Disagree', 'Neutral', 'Agree']
internal codes: [0, 1, 2]
numeric values: [10, 20, 30]


## Note 3: unused categories remain

Use the lecture pattern and inspect the result before continuing.

In [36]:
# Filter to Havering
havering = venues.loc[venues["borough_name"] == "Havering"]

# How many possible categories? 
print("possible labels:", len(havering["venue_type"].cat.categories))

# How many actual categories in the Havering data?
print("observed types:", havering["venue_type"].nunique())

# And if you want to remove those categories
havering["venue_type"] = havering["venue_type"].cat.remove_unused_categories()
print("labels after removal:", len(havering["venue_type"].cat.categories))

possible labels: 26
observed types: 9
labels after removal: 9


## The borough data for our bands

One row in borough_context is one area.

In [7]:
# Load borough_context and inspect the prepared record-rate column.
borough_context = pd.read_csv(
    data_path("london_cultural_infrastructure_2023_borough_context.csv")
)
borough_context[["borough_name", "venues_per_10k_residents"]].head()

,borough_name,venues_per_10k_residents
0,Westminster,28.32
1,Camden,20.36
2,Hackney,15.34
3,Southwark,9.95
4,Tower Hamlets,10.38


## Ordered categories (i.e., ordinal variables)

Use the familiar .loc conditional assignment pattern, then convert to category. .cat.set_categories sets the label list; ordered=True makes its order meaningful. Low is below 5, Medium is 5 to below 15, and High is 15 or above.

In [37]:
# Create rate_band that is a categorical variable represenitng the number of cultural venues

# Create a new column for our ordinal variable (in this case, it's just an empty string to start)
borough_context["rate_band"] = ""

# Use the existing date (venues_per_10k_residents) to assign the rating to the new column
# We decide what the thresholds are for our categories - they can be whatever we want!
# In this case we're using less than 5, between 5 and 14, and 15 or greater

# Set low
borough_context.loc[borough_context["venues_per_10k_residents"] < 5, "rate_band"] = "Low"
# Set medium
borough_context.loc[(borough_context["venues_per_10k_residents"] >= 5) 
    & (borough_context["venues_per_10k_residents"] < 15), "rate_band"] = "Medium"
# Set high
borough_context.loc[borough_context["venues_per_10k_residents"] >= 15, "rate_band"] = "High"

# Turn the column into a categorical dtype and make them ordered
borough_context["rate_band"] = borough_context["rate_band"].astype("category")
borough_context["rate_band"] = borough_context["rate_band"].cat.set_categories(
    ["Low", "Medium", "High"], ordered=True
)

# Examine
print(borough_context["rate_band"].cat.categories)
print("ordered:", borough_context["rate_band"].cat.ordered)

Index(['Low', 'Medium', 'High'], dtype='str')
ordered: True


## The same rates, different bands

q=3 asks qcut for three approximately equal-sized groups. retbins=True returns both the labels and boundaries: keep the labels in quantile_band and boundaries in quantile_edges. The next slide interprets these results without repeating the calculation.

In [ ]:
# Make quantile bands from the same rate column. Compare group sizes under the two banding rules.
borough_context["quantile_band"], quantile_edges = pd.qcut(
    borough_context["venues_per_10k_residents"],
    q=3, labels=["Low", "Medium", "High"], retbins=True,
)

# Comapre the two
band_counts = pd.DataFrame({
    "Fixed thresholds": borough_context["rate_band"].value_counts(sort=False),
    "Quantile bands": borough_context["quantile_band"].value_counts(sort=False),
})
print(band_counts)

        Fixed thresholds  Quantile bands
Low                   22              11
Medium                 7              11
High                   4              11


## Our bike-hire extract: one record per hire

Switch from one borough per record to one recorded hire per record. 

Note:

* The source covers 16–31 July 2025, not the full year. 
* Each hire has endpoint labels, calendar fields, bike model and recorded duration in minutes.

In [10]:
# Load the supplied bike-hire CSV as trips at this point in the lecture, then preview it.
trips = pd.read_csv(data_path("tfl-bike-hire/bike_trips_16-31Jul2025_with_borough.csv"))
trips.head()

,trip_id,start_time,date,weekday,start_station,start_borough,end_station,end_borough,bike_model,duration_min
0,150869175,2025-07-31 23:59:00,2025-07-31,Thursday,"Lambeth North Station, Waterloo",Lambeth,"Kennington Road Post Office, Oval",Lambeth,CLASSIC,5.5
1,150869176,2025-07-31 23:59:00,2025-07-31,Thursday,"Falkirk Street, Hoxton",Hackney,"Hoxton Street, Hoxton",Hackney,CLASSIC,12.4
2,150869177,2025-07-31 23:59:00,2025-07-31,Thursday,"Breams Buildings, Holborn",City of London,"Hardwick Street, Clerkenwell",Islington,CLASSIC,8.6
3,150869168,2025-07-31 23:58:00,2025-07-31,Thursday,"Warren Street Station, Euston",Camden,"Killick Street, King's Cross",Islington,CLASSIC,17.3
4,150869169,2025-07-31 23:58:00,2025-07-31,Thursday,"British Museum, Bloomsbury",Camden,"Haggerston Road, Haggerston",Hackney,CLASSIC,18.3


## Create a temporal category: weekday or weekend?

Are recorded hires longer on weekdays or at weekends? First create the grouping variable using the supplied weekday labels, familiar .loc assignments and | for or. Weekday/weekend is nominal. We will compare duration in Part III.

In [ ]:
# Create day_type from the existing weekday labels with .loc and |, then convert it to category.

# Create a new column and populate it with "Weekday" for every row
trips["day_type"] = "Weekday"

# Use conditionals to replace "Weekday" with "Weekend" for Saturdays and Sundays.
trips.loc[
    (trips["weekday"] == "Saturday") | (trips["weekday"] == "Sunday"),
    "day_type",
] = "Weekend"

# Turn the column into a "category" datatype
trips["day_type"] = trips["day_type"].astype("category")

# Check the data
print(trips["day_type"].cat.categories)

Index(['Weekday', 'Weekend'], dtype='str')


# Part 2: Tables

## Frequency tables and the mode

The mode is the most frequent label. idxmax returns the label attached to the largest count, here Archives.

In [ ]:
# Count venue-type labels, display six counts and identify the modal label.

# Saving a very simple table as type_counts created with .value_counts()
type_counts = venues["venue_type"].value_counts()
print(type_counts.head(6))

# "Averages" with categorical data: the mode
print("mode:", type_counts.idxmax())

venue_type
Archives                544
Music venues all        371
Libraries               347
Commercial galleries    300
Artists workspaces      270
Theatre                 266
Name: count, dtype: int64
mode: Archives


## Counts and percentages

`normalize=True` divides by all included records; multiply by 100 for percentages. Displaying six labels does not limit the denominator to six types.

In [38]:
# Express venue-type counts as percentages and round the display.
type_percent = venues["venue_type"].value_counts(normalize = True) * 100
print(type_percent.round(1).head(6))

venue_type
Archives                12.3
Music venues all         8.4
Libraries                7.8
Commercial galleries     6.8
Artists workspaces       6.1
Theatre                  6.0
Name: proportion, dtype: float64


## Use the weekday/weekend category: how many hires?

The category created in Part I has 358,643 weekday and 84,896 weekend hires. The extract contains 12 weekdays and four weekend days; raw totals therefore cover different numbers of dates.

In [14]:
# Count day_type labels and calculate their percentages across the full trip extract.
print(trips["day_type"].value_counts())
print((trips["day_type"].value_counts(normalize=True) * 100).round(1))

day_type
Weekday    358643
Weekend     84896
Name: count, dtype: int64
day_type
Weekday    80.9
Weekend    19.1
Name: proportion, dtype: float64


## A readable borough by type crosstab

The first crosstab argument supplies rows and the second supplies columns. Calculate the full table, then select only the display.

In [ ]:
# Choose the four boroughs for display. Choose the three venue types for display. Make the full London crosstab, then select the display.

# Create a crosstab of boroughs and venue types
borough_type_counts = pd.crosstab(
    venues["borough_name"], venues["venue_type"],
)

# Rather than displaying the entire crosstab (which would be 33x26) we choose 3 boroughs and 3 venue types to show
shown_boroughs = ["Camden", "Hackney", "Westminster", "Havering"]
shown_types = ["Libraries", "Music venues all", "Music nightclubs"]
borough_type_counts.loc[shown_boroughs, shown_types]

venue_type,Libraries,Music venues all,Music nightclubs
borough_name,,,
Camden,12,36,9
Hackney,9,71,31
Westminster,11,52,25
Havering,10,0,0


## Crosstab with margins

Margins are totals. Hidden types and boroughs still contribute to them; the grand total counts 4,434 records.

In [16]:
# Add Total margins to the full crosstab, then display the chosen boroughs and types with totals.
with_totals = pd.crosstab(
    venues["borough_name"], venues["venue_type"],
    margins=True, margins_name="Total",
)
with_totals.loc[shown_boroughs + ["Total"], shown_types + ["Total"]]

venue_type,Libraries,Music venues all,Music nightclubs,Total
borough_name,,,,
Camden,12,36,9,469
Hackney,9,71,31,403
Westminster,11,52,25,653
Havering,10,0,0,22
Total,347,371,133,4434


## Row percentages: Camden’s venue mix

Camden has 36 "Music venues all" records out of the 469 recorded venues in the borough (that is, in Camden specifically) of every type: that is 7.7% of its venues are Music venues.

Using `normalize="index"` we can report this proportion.

In [17]:
# Calculate row percentages using every type, then display the chosen cells.
row_percent = pd.crosstab(
    venues["borough_name"], venues["venue_type"],
    normalize="index", 
) * 100
row_percent.loc[shown_boroughs, shown_types].round(1)

venue_type,Libraries,Music venues all,Music nightclubs
borough_name,,,
Camden,2.6,7.7,1.9
Hackney,2.2,17.6,7.7
Westminster,1.7,8.0,3.8
Havering,45.5,0.0,0.0


## Column percentages: music venues across London

In contrast, Camden has 36 of all of London's 371 "Music venues all" records. When counted this way, Camden has 9.7% of all the meusic venues in London.

To calculate this we use `normalize="columns"`.

In [18]:
# Calculate column percentages using all London boroughs, then display the chosen cells.
column_percent = pd.crosstab(
    venues["borough_name"], venues["venue_type"],
    normalize="columns",
) * 100
column_percent.loc[shown_boroughs, shown_types].round(1)

venue_type,Libraries,Music venues all,Music nightclubs
borough_name,,,
Camden,3.5,9.7,6.8
Hackney,2.6,19.1,23.3
Westminster,3.2,14.0,18.8
Havering,2.9,0.0,0.0


# Part 3: Grouping

## Our grouping question: where do hires start, and how long do they last?

Reusing the day_type created data, follow the four decisions: group, measurement, calculation, interpretation.

In [19]:
# Inspect the supplied trips table. Identify the grouping and numerical columns.
trips[["start_borough", "end_borough", "day_type", "duration_min"]].head(3)

,start_borough,end_borough,day_type,duration_min
0,Lambeth,Lambeth,Weekday,5.5
1,Hackney,Hackney,Weekday,12.4
2,City of London,Islington,Weekday,8.6


## Step 1: group hires by starting borough

groupby defines the groups; size counts their records. Display every starting borough, as on the current slide.

In [20]:
# Create borough_groups and print the size of every starting-borough group.
# Create the "groupby" obeject
borough_groups = trips.groupby("start_borough")

# Print the size of each category
print(borough_groups.size())

start_borough
Camden                     39135
City of London             28988
Hackney                    20434
Hammersmith and Fulham     24390
Islington                  18256
Kensington and Chelsea     44598
Lambeth                    32875
Newham                      4400
Southwark                  32810
Tower Hamlets              60677
Wandsworth                 25706
Westminster               111270
dtype: int64


## Steps 2 and 3: select duration and calculate its median

Reuse borough_groups. Select duration_min, calculate the median within each group and display the full result in minutes.

In [21]:
# Calculate and display median duration for every starting borough.
# Now we can use .median() to calculate the median duration time.
median_duration = borough_groups["duration_min"].median()
print(median_duration)

start_borough
Camden                    13.4
City of London            13.1
Hackney                   12.3
Hammersmith and Fulham    12.7
Islington                 11.8
Kensington and Chelsea    14.3
Lambeth                   13.0
Newham                    24.0
Southwark                 13.4
Tower Hamlets             12.3
Wandsworth                14.1
Westminster               16.6
Name: duration_min, dtype: float64


## EDA normally reveals awkward values

Inspect describe's minimum, quartiles, maximum and other summaries. What seems unusual, and what would you investigate? Keep the original values.

In [22]:
# Use describe to inspect the distribution of recorded durations. Identify values that need investigation.
# Use .describe to get an overview of the data distribution for trip durations
print(trips["duration_min"].describe().round(1))

count    443539.0
mean         23.0
std         231.3
min           0.0
25%           8.1
50%          14.0
75%          23.0
max       75882.8
Name: duration_min, dtype: float64


## EDA normally reveals awkward values II

Use comparisons and sum to count zero durations and those over 1,440 minutes. Unusual values are normal EDA findings to investigate; these checks do not establish their cause or justify silently deleting them.

In [23]:
# Count zero durations and durations over 24 hours without changing the records.
# There are clear instances of trips of 
# zero length and longer than 24 hours.
print("Zero durations:", 
    (trips["duration_min"] == 0).sum())
print("Over 24 hours:", 
    (trips["duration_min"] > 1440).sum())

Zero durations: 199
Over 24 hours: 269


## Compare count, mean, and median in one table

agg applies the listed calculations to duration within each borough. count is non-missing hires; sum is total recorded hire-minutes; mean and median are minutes per recorded hire. All supplied values, including extreme durations, remain in the calculation.

In [24]:
# Summarise duration within every starting borough using count, sum, mean and median.
duration_summary = borough_groups["duration_min"].agg(["count", "sum", "mean", "median"])
duration_summary.round(1)

,count,sum,mean,median
start_borough,,,,
Camden,39135,834821.8,21.3,13.4
City of London,28988,521856.0,18.0,13.1
Hackney,20434,415807.1,20.3,12.3
Hammersmith and Fulham,24390,516841.4,21.2,12.7
Islington,18256,297992.1,16.3,11.8
Kensington and Chelsea,44598,1116005.5,25.0,14.3
Lambeth,32875,664102.0,20.2,13.0
Newham,4400,183944.9,41.8,24.0
Southwark,32810,702487.5,21.4,13.4


## Reuse the time category: compare hire durations

Change only the grouping column to day_type. The numerical measure remains recorded duration per hire. Weekday median is 13.7 minutes; weekend median is 15.2. All original duration values are retained.

In [25]:
# Group by the existing day_type category and summarise the same duration values with count, mean and median.
day_type_summary = trips.groupby("day_type")["duration_min"].agg(["count", "mean", "median"])
day_type_summary.round(1)

,count,mean,median
day_type,,,
Weekday,358643,22.1,13.7
Weekend,84896,26.8,15.2


## Group by two variables: starting and ending borough

The two-column list defines a group for each directed endpoint pair. Sort the full result as on the current slide. Endpoints do not establish the actual route, rider identity or trip purpose.

In [26]:
# Count directed starting/ending borough pairs and sort the full result largest first.
borough_flows = trips.groupby(["start_borough", "end_borough"]).size()
print(borough_flows.sort_values(ascending=False))

start_borough           end_borough           
Westminster             Westminster               55685
Tower Hamlets           Tower Hamlets             35869
Kensington and Chelsea  Kensington and Chelsea    17531
                        Westminster               13846
Westminster             Kensington and Chelsea    13563
                                                  ...  
Newham                  Wandsworth                    7
Kensington and Chelsea  Newham                        7
Wandsworth              Newham                        4
Hammersmith and Fulham  Newham                        3
Newham                  Hammersmith and Fulham        1
Length: 144, dtype: int64
